In [ ]:
!pip install odfpy

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_excel('MESSY_CARS.ods', engine='odf')
df.head()

---
## Step 0 — First Look at the Data
Before fixing anything, we look at everything first.
Goal: understand what columns exist and what types they are.

In [ ]:
# what columns do we have and what is their data type?
df.dtypes
# 'object' means text/string
# year, selling_price, km_driven should be numbers but they show 'object'
# that means they have messy text inside them

In [ ]:
# how many rows and columns?
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
# are there any empty/missing rows?
df.isnull().sum()
# this only catches truly empty rows
# values like 'N/A' 'unknown' 'Ask for price' are TEXT so they won't show here
# we will catch those one by one when we clean each column

---
## Step 1 — Duplicates
We check for duplicate rows FIRST before cleaning anything.
Why first? Because if we clean first and THEN remove duplicates,
we waste time cleaning rows that will be deleted anyway.

### Identifying duplicate rows

In [ ]:
# how many duplicate rows are there?
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# show the actual duplicate rows so we can see them
df[df.duplicated()]

### Solving — remove duplicate rows

In [ ]:
# keep='first' means: keep the first time a row appears, delete the rest
df = df.drop_duplicates(keep='first')

# reset_index fixes the row numbers after deleting rows
# without this: row numbers have gaps like 0,1,3,5 (2 and 4 deleted)
# with this:    row numbers are clean again 0,1,2,3
# drop=True means: don't save the old broken numbers as a new column
df = df.reset_index(drop=True)

print("Rows after removing duplicates:", df.shape[0])

---
## Step 2 — Column: name
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'name' column

In [ ]:
# look at all unique values
df['name'].unique()

In [ ]:

# issues we can see:
# 1. some names have extra spaces at start or end
# 2. inconsistent casing
# 3. some names are null

# count null values
print("Empty names:", df['name'].isnull().sum())

### Solving issues in 'name' column

In [ ]:
# ISSUE 1 FIX: remove extra spaces from start and end
print("Before strip:")
print(df['name'].unique())

df['name'] = df['name'].str.strip()

print()
print("After strip:")
print(df['name'].unique())

In [ ]:
# ISSUE 2 FIX: fix the casing - make everything Title Case
# Title Case = first letter of each word is capital, rest are small
print("Before title case:")
print(df['name'].unique())

df['name'] = df['name'].str.title()

print()
print("After title case:")
print(df['name'].unique())

In [ ]:
# ISSUE 3 FIX: empty names -> fill with 'Unknown'
# now fill NaN with 'Unknown'
df['name'] = df['name'].fillna('Unknown')

print("Any missing names left?", df['name'].isnull().sum())

In [ ]:
# final check - name column looks clean now
print("All unique name values after cleaning:")
df['name'].unique()

In [ ]:
df['name'].isnull().sum()

---
## Step 3 — Column: year
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'year' column

In [ ]:
# look at all unique values in year column
df['year'].unique()

In [ ]:
# checking if thers any value ending with .0
df["year"].astype(str).str.endswith(".0", na=False).any()

In [ ]:
# value count containing .
[df["year"].astype(str).str.contains(".", regex=False, na=False).sum()]

In [ ]:
# rows containing .
df.loc[df["year"].astype(str).str.contains(".", regex=False, na=False), "year"]

In [ ]:
# issues we can see:
# 1. data type is 'object' (text) - should be a number
# 2. some values have extra spaces like '  2019  '
# 3. some values are text like '??', 'year unknown', ''
# 4. some years are impossible like 1800, 1850 (cars didn't exist then; they invented  in 1885)
# 5. null values

print("Current data type of year column:", df['year'].dtype)
print("Missing values:", df['year'].isnull().sum())

### Solving issues in 'year' column

In [ ]:
# ISSUE 1 and 3 FIX: convert text to number
# pd.to_numeric tries to convert every value to a number
# errors='coerce' means:
#   if a value CANNOT be converted (like 'N/A', '??', 'year unknown')
#   instead of crashing, just make it NaN (missing)


print("Before:")
print(df['year'].unique())

df['year'] = pd.to_numeric(df['year'], errors='coerce')

print()
print("After pd.to_numeric:")
print(df['year'].unique())
print()
print("Data type now:", df['year'].dtype)
print("Missing values now:", df['year'].isnull().sum())

In [ ]:
# ISSUE 4 FIX: remove impossible years
# first car was made in 1885
# we also don't accept future years above 2026
# anything outside this range is a data entry mistake -> make it NaN

print("Impossible years found:")
print(df['year'][(df['year'] < 1885) | (df['year'] > 2026)].values)

# set impossible years to NaN
df.loc[df['year'] < 1886, 'year'] = np.nan
df.loc[df['year'] > 2024, 'year'] = np.nan

print()
print("Missing values after removing impossible years:", df['year'].isnull().sum())

In [ ]:
# now fill missing years with the median year or better to compare first (models and pricing)
# median = the middle value when all years are sorted
# we use median instead of mean because
# mean gets pulled by extreme values, median does not

median_year = df['year'].median()
print("Median year:", median_year)

df['year'] = df['year'].fillna(median_year)

print("Missing values after fillna:", df['year'].isnull().sum())

In [ ]:
df.dtypes

In [ ]:
# finally convert to integer
# year should be 2018 not 2018.0
# we can only do astype(int) AFTER filling all NaN
# because integer columns cannot hold NaN in pandas

df['year'] = df['year'].astype(int)

print("Data type after astype(int):", df['year'].dtype)
print()
print("All year values after cleaning:")
print(df['year'].unique())

---
## Step 4 — Column: selling_price
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'selling_price' column

In [ ]:
# look at all unique values
df['selling_price'].unique()

In [ ]:
# issues we can see:
# 1. currency symbols: 'Rs.' and '₹' need to be removed
# 2. comma separators: '5,50,000' needs to become '550000'
# 3. text values: 'Ask for price', 'On Request', 'N/A' - cannot be numbers
# 4. negative prices like '-350000' - impossible, price can't be negative
# 5. extra spaces
# 6. data type is 'object' - needs to become a number

print("Data type:", df['selling_price'].dtype)
print("Missing values:", df['selling_price'].isnull().sum())

### Solving issues in 'selling_price' column

In [ ]:
# ISSUE 5 FIX: remove spaces first
df['selling_price'] = df['selling_price'].str.strip()
print("Spaces removed")

In [ ]:
# ISSUE 1 FIX: remove currency symbols
# str.replace('what to find', 'what to replace with')
# we replace with '' which means delete it
# regex=False means treat 'Rs.' as plain text, not a pattern
# (the dot in 'Rs.' is a special character in regex so we turn regex off)

print("Before removing symbols:")
print(df['selling_price'].unique())

df['selling_price'] = df['selling_price'].str.replace('Rs.', '', regex=False)
df['selling_price'] = df['selling_price'].str.replace('₹', '', regex=False)

# strip again because 'Rs. 550000' after removing 'Rs.' becomes ' 550000' (space left)
df['selling_price'] = df['selling_price'].str.strip()

print()
print("After removing symbols:")
print(df['selling_price'].unique())

In [ ]:
# ISSUE 2 FIX: remove comma separators
# '5,50,000' -> '550000'
print("Before removing commas:")
print(df['selling_price'].unique())

df['selling_price'] = df['selling_price'].str.replace(',', '', regex=False)

print()
print("After removing commas:")
print(df['selling_price'].unique())

In [ ]:
# ISSUE 3 FIX: convert to number - text like 'Ask for price' becomes NaN
print("Before to_numeric:")
print(df['selling_price'].unique())

df['selling_price'] = pd.to_numeric(df['selling_price'], errors='coerce')

print()
print("After to_numeric:")
print(df['selling_price'].unique())
print()
print("Data type now:", df['selling_price'].dtype)
print("Missing values now:", df['selling_price'].isnull().sum())

In [ ]:
# # ISSUE 4 FIX: remove negative prices
# # a car price cannot be 0 or negative - it's a data entry mistake
# print("Negative or zero prices found:")
# print(df[df['selling_price'] <= 0]['selling_price'].values) # none found

# df.loc[df['selling_price'] <= 0, 'selling_price'] = np.nan # if found convert to nan

# print()
# print("Missing values after removing negatives:", df['selling_price'].isnull().sum())

In [ ]:
# now fill missing prices with the median price
median_price = df['selling_price'].median()
print("Median price:", median_price)

df['selling_price'] = df['selling_price'].fillna(median_price)

# convert to integer - price is whole rupees, no decimals needed
df['selling_price'] = df['selling_price'].astype(int)

print()
print("Data type after astype(int):", df['selling_price'].dtype)
print("Sample values after cleaning:")
print(df['selling_price'].unique()[:10])

---
## Step 5 — Column: km_driven
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'km_driven' column

In [ ]:
# look at all unique values
df['km_driven'].unique()

In [ ]:
# issues we can see:
# 1. unit text attached: '45,000 km' - need to remove ' km'
# 2. comma separators: '45,000' -> '45000'
# 3. text values: 'unknown', 'N/A' - cannot be numbers
# 4. negative values: impossible, km cannot be negative (not present in this data)
# 5. unrealistically high: above 500000 is likely a typo (not present in this data)
# 6. extra spaces
# 7. data type is 'object' - needs to become a number

print("Data type:", df['km_driven'].dtype)
print("Missing values:", df['km_driven'].isnull().sum())

### Solving issues in 'km_driven' column

In [ ]:
# ISSUE 6 FIX: remove spaces first
df['km_driven'] = df['km_driven'].str.strip()
print("Spaces removed")

In [ ]:
# ISSUE 1 FIX: remove ' km' unit text
# '45,000 km' -> '45,000'
print("Before removing km unit:")
print(df['km_driven'].unique())

df['km_driven'] = df['km_driven'].str.replace(' km', '', regex=False)
df['km_driven'] = df['km_driven'].str.replace('km', '', regex=False)

# strip again in case removing 'km' left a trailing space
df['km_driven'] = df['km_driven'].str.strip()

print()
print("After removing km unit:")
print(df['km_driven'].unique())

In [ ]:
# ISSUE 2 FIX: remove comma separators
# '45,000' -> '45000'
df['km_driven'] = df['km_driven'].str.replace(',', '', regex=False)

print("After removing commas:")
print(df['km_driven'].unique())

In [ ]:
# ISSUE 3 FIX: convert to number - 'unknown', 'N/A' become NaN
df['km_driven'] = pd.to_numeric(df['km_driven'], errors='coerce')

print("Data type now:", df['km_driven'].dtype)
print("Missing values now:", df['km_driven'].isnull().sum())

In [ ]:
# ISSUE 4 FIX: remove negative km values
print("Negative km values found:")
print(df[df['km_driven'] < 0]['km_driven'].values)

df.loc[df['km_driven'] < 0, 'km_driven'] = np.nan
print("Negatives set to NaN")

In [ ]:
# ISSUE 5 FIX: remove unrealistically high km
# more than 500,000 km for a used car is almost certainly a typo
print("Unrealistically high km found (above 500000):")
print(df[df['km_driven'] > 500000]['km_driven'].values)

df.loc[df['km_driven'] > 500000, 'km_driven'] = np.nan
print("Unrealistic values set to NaN")

In [ ]:
# fill missing km with median, then convert to integer
median_km = df['km_driven'].median()
print("Median km:", median_km)

df['km_driven'] = df['km_driven'].fillna(median_km)
df['km_driven'] = df['km_driven'].astype(int)

print()
print("Data type after cleaning:", df['km_driven'].dtype)
print("Sample values after cleaning:")
print(df['km_driven'].unique()[:10])

---
## Step 6 — Column: fuel
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'fuel' column

In [ ]:
# look at all unique values
df['fuel'].unique()

In [ ]:
# issues we can see:
# 1. inconsistent casing: 'Petrol', 'PETROL', 'petrol' all mean the same thing
# 2. some values are empty/missing

print("Missing values:", df['fuel'].isnull().sum())
print("Empty string values:", (df['fuel'].str.strip() == '').sum())

### Solving issues in 'fuel' column

In [ ]:
# ISSUE 1 FIX: strip spaces and make Title Case
# 'PETROL' -> 'Petrol'
# 'diesel' -> 'Diesel'
# 'petrol' -> 'Petrol'
print("Before:")
print(df['fuel'].unique())

df['fuel'] = df['fuel'].str.strip()
df['fuel'] = df['fuel'].str.title()

print()
print("After title case:")
print(df['fuel'].unique())

In [ ]:
# ISSUE 2 FIX: fill missing values with the mode
# mode = the value that appears most often

# first replace empty strings with NaN so fillna can work
df['fuel'] = df['fuel'].replace('', np.nan)

# find the most common fuel type
most_common_fuel = df['fuel'].mode()[0]
print("Most common fuel type:", most_common_fuel)

# fill missing values with it
df['fuel'] = df['fuel'].fillna(most_common_fuel)

print("Missing values left:", df['fuel'].isnull().sum())
print()
print("Final unique values:")
print(df['fuel'].unique())

---
## Step 7 — Column: seller_type
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'seller_type' column

In [ ]:
# look at all unique values
df['seller_type'].unique()

In [ ]:
# issues we can see:
# 1. inconsistent casing: 'Individual', 'DEALER', 'dealer'
# 2. some values are empty/missing

print("Missing values:", df['seller_type'].isnull().sum())
print("Empty string values:", (df['seller_type'].str.strip() == '').sum())

### Solving issues in 'seller_type' column

In [ ]:
# ISSUE 1 FIX: strip spaces and make Title Case
print("Before:")
print(df['seller_type'].unique())

df['seller_type'] = df['seller_type'].str.strip()
df['seller_type'] = df['seller_type'].str.title()

print()
print("After title case:")
print(df['seller_type'].unique())

In [ ]:
# ISSUE 2 FIX: fill missing values with mode
df['seller_type'] = df['seller_type'].replace('', np.nan)

most_common_seller = df['seller_type'].mode()[0]
print("Most common seller type:", most_common_seller)

df['seller_type'] = df['seller_type'].fillna(most_common_seller)

print("Missing values left:", df['seller_type'].isnull().sum())
print()
print("Final unique values:")
print(df['seller_type'].unique())

---
## Step 8 — Column: transmission
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'transmission' column

In [ ]:
# look at all unique values
df['transmission'].unique()

In [ ]:
# issues we can see:
# 1. inconsistent casing: 'Manual', 'MANUAL', 'manual'
# 2. abbreviation: 'Auto' should be 'Automatic'
# 3. 'man' should be 'Manual'
# 3. some values are empty/missing

print("Missing values:", df['transmission'].isnull().sum())
print("Empty string values:", (df['transmission'].str.strip() == '').sum())

### Solving issues in 'transmission' column

In [ ]:
# ISSUE 1 FIX: strip spaces and make Title Case
print("Before:")
print(df['transmission'].unique())

df['transmission'] = df['transmission'].str.strip()
df['transmission'] = df['transmission'].str.title()

print()
print("After title case:")
print(df['transmission'].unique())

In [ ]:
# ISSUE 2 and 3 FIX: replace 'Auto' with 'Automatic' and ''Man' with 'Manual'
# .replace() takes a dictionary {old value : new value}
# it looks at every cell and if it finds 'Auto', replaces it with 'Automatic'
# everything else stays unchanged

print("Before replace:")
print(df['transmission'].unique())

df['transmission'] = df['transmission'].replace({'Auto': 'Automatic'})
df['transmission'] = df['transmission'].replace({'Man': 'Manual'})


print()
print("After replace:")
print(df['transmission'].unique())

In [ ]:
# ISSUE 3 FIX: fill missing values with mode
df['transmission'] = df['transmission'].replace('', np.nan)

most_common_transmission = df['transmission'].mode()[0]
print("Most common transmission:", most_common_transmission)

df['transmission'] = df['transmission'].fillna(most_common_transmission)

print("Missing values left:", df['transmission'].isnull().sum())
print()
print("Final unique values:")
print(df['transmission'].unique())

---
## Step 9 — Column: owner
Let's look at it, find all problems, then fix them one by one.

### Identifying issues in 'owner' column

In [ ]:
# look at all unique values
df['owner'].unique()

In [ ]:
# issues we can see:
# 1. inconsistent casing: 'First Owner', 'SECOND OWNER', 'first owner'
# 2. abbreviations: '1st owner' should be 'First Owner', '2nd owner' -> 'Second Owner'
# 3. some values are empty/missing

print("Missing values:", df['owner'].isnull().sum())
print("Empty string values:", (df['owner'].str.strip() == '').sum())

### Solving issues in 'owner' column

In [ ]:
# ISSUE 1 FIX: strip spaces and make Title Case
print("Before:")
print(df['owner'].unique())

df['owner'] = df['owner'].str.strip()
df['owner'] = df['owner'].str.title()

print()
print("After title case:")
print(df['owner'].unique())

In [ ]:
# ISSUE 2 FIX: replace abbreviations with correct full names
# IMPORTANT: we do this AFTER title case
# because '1st owner' becomes '1St Owner' after .str.title()
# so we need to match against '1St Owner' not '1st owner'

print("Before replace:")
print(df['owner'].unique())

df['owner'] = df['owner'].replace({
    '1St Owner': 'First Owner',
    '2Nd Owner': 'Second Owner'
})

print()
print("After replace:")
print(df['owner'].unique())

In [ ]:
# ISSUE 3 FIX: fill missing values with mode
df['owner'] = df['owner'].replace('', np.nan)

most_common_owner = df['owner'].mode()[0]
print("Most common owner type:", most_common_owner)

df['owner'] = df['owner'].fillna(most_common_owner)

print("Missing values left:", df['owner'].isnull().sum())
print()
print("Final unique values:")
print(df['owner'].unique())

---
## Step 10 — Final Check
After cleaning every column, we do a final check to confirm everything is correct.

In [ ]:
# check data types - all should be correct now
print("=== DATA TYPES ===")
print(df.dtypes)
print()
print("What we expect:")
print("name          -> object  (text)  ✓")
print("year          -> int64   (whole number) ✓")
print("selling_price -> int64   (whole number) ✓")
print("km_driven     -> int64   (whole number) ✓")
print("fuel          -> object  (text)  ✓")
print("seller_type   -> object  (text)  ✓")
print("transmission  -> object  (text)  ✓")
print("owner         -> object  (text)  ✓")

In [ ]:
# check missing values - all should be 0 now
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
print(missing)
print()
if missing.sum() == 0:
    print("No missing values anywhere!")
else:
    print("Some missing values remain - check the columns above")

In [ ]:
# check value ranges for numeric columns
print("=== NUMERIC COLUMN RANGES ===")
print()
print("year:          min =", df['year'].min(), "  max =", df['year'].max())
print("selling_price: min =", df['selling_price'].min(), "  max =", df['selling_price'].max())
print("km_driven:     min =", df['km_driven'].min(), "  max =", df['km_driven'].max())
print()
print("What we expect:")
print("year:          between 1886 and 2024")
print("selling_price: all positive numbers")
print("km_driven:     all positive, max below 500000")

In [ ]:
# check categorical columns - should have clean consistent values
print("=== CATEGORICAL COLUMN VALUES ===")
for col in ['fuel', 'seller_type', 'transmission', 'owner']:
    print()
    print(f"{col}:")
    print(df[col].value_counts())